# SNAPflow StaMPS Prep Notebook

This notebook downloads two Sentinel-1 SLC products with `phidown`, prepares them with `snapflow`, and runs the StaMPS-prep chain:

- `split_orbit`
- `coregistration`
- `deburst`
- `interferogram`
- `topo_phase_removal`
- `subset`
- `terrain_correction`

The `coregistration` step is handled through the new pair-aware `GPT.topsar_coregistration(...)` helper, which generates a temporary SNAP XML graph because `Back-Geocoding` needs both a master and a slave input.

Runtime note: for TOPSAR/ESD, SNAP needs burst-level products during coregistration, so the executable order is `split_orbit -> coregistration -> deburst -> ...`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

from dotenv import load_dotenv

from sarpyx.snapflow.engine import GPT
from sarpyx.snapflow.snap2stamps import PairProducts, prepare_pair, run_pair_workflow

load_dotenv()

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "output" / "insar_stamps_prep"
CONFIG_FILE = REPO_ROOT / ".s5cfg"
GPT_PATH = Path(os.getenv("GPT_PATH", "/usr/local/snap/bin/gpt"))
SNAP_USERDIR = Path(os.getenv("SNAP_USERDIR", REPO_ROOT / ".snap"))

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SNAP_USERDIR.mkdir(parents=True, exist_ok=True)

assert CONFIG_FILE.exists(), f"Missing phidown config: {CONFIG_FILE}"
print(json.dumps({
    "repo_root": REPO_ROOT.as_posix(),
    "data_dir": DATA_DIR.as_posix(),
    "output_dir": OUTPUT_DIR.as_posix(),
    "config_file": CONFIG_FILE.as_posix(),
    "gpt_path": GPT_PATH.as_posix(),
    "snap_userdir": SNAP_USERDIR.as_posix(),
}, indent=2))

{
  "repo_root": "/shared/home/rdelprete/PythonProjects/srp",
  "data_dir": "/shared/home/rdelprete/PythonProjects/srp/data",
  "output_dir": "/shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep",
  "config_file": "/shared/home/rdelprete/PythonProjects/srp/.s5cfg",
  "gpt_path": "/shared/home/rdelprete/PythonProjects/srp/snap13/bin/gpt",
  "snap_userdir": "/shared/home/rdelprete/PythonProjects/srp/.snap"
}


## Product Definitions

In [2]:
PRODOTTO1 = "S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE.SAFE"
PRODOTTO2 = "S1C_IW_SLC__1SDV_20260208T122502_20260208T122529_006264_00C961_E47A.SAFE"

PRODUCT_NAMES = [PRODOTTO1, PRODOTTO2]
PRODUCT_PATHS = [DATA_DIR / name for name in PRODUCT_NAMES]

MASTER_PRODUCT = DATA_DIR / PRODOTTO1
SLAVE_PRODUCT = DATA_DIR / PRODOTTO2

print("Master:", MASTER_PRODUCT)
print("Slave :", SLAVE_PRODUCT)

Master: /shared/home/rdelprete/PythonProjects/srp/data/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE.SAFE
Slave : /shared/home/rdelprete/PythonProjects/srp/data/S1C_IW_SLC__1SDV_20260208T122502_20260208T122529_006264_00C961_E47A.SAFE


## Download With `phidown`

The cell below skips products that already exist locally.

In [3]:
for product_name, product_path in zip(PRODUCT_NAMES, PRODUCT_PATHS):
    if product_path.exists():
        print(f"Skipping download, already present: {product_path.name}")
        continue

    cmd = [
        sys.executable,
        "-m",
        "phidown",
        "--name",
        product_name,
        "-o",
        DATA_DIR.as_posix(),
        "-c",
        CONFIG_FILE.as_posix(),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

for path in PRODUCT_PATHS:
    print(path, "exists=", path.exists())

Skipping download, already present: S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE.SAFE
Skipping download, already present: S1C_IW_SLC__1SDV_20260208T122502_20260208T122529_006264_00C961_E47A.SAFE
/shared/home/rdelprete/PythonProjects/srp/data/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE.SAFE exists= True
/shared/home/rdelprete/PythonProjects/srp/data/S1C_IW_SLC__1SDV_20260208T122502_20260208T122529_006264_00C961_E47A.SAFE exists= True


## Processing Configuration

These defaults are chosen to complete in this VM: a small burst range is used during `TOPSAR-Split`, and the later `subset` step uses pixel coordinates instead of a scene-specific geographic polygon.

In [4]:
SUBSWATH = "IW2"
POLARISATIONS = ["VV"]
FIRST_BURST_INDEX = 1
LAST_BURST_INDEX = 3
SUBSET_REGION = "0,0,4096,4096"

DEM_NAME = "Copernicus 30m Global DEM"
PIXEL_SPACING_M = 14.0

COMMON_OVERRIDES = {
    "topsar_split": {
        "subswath": SUBSWATH,
        "selected_polarisations": POLARISATIONS,
        "first_burst_index": FIRST_BURST_INDEX,
        "last_burst_index": LAST_BURST_INDEX,
    },
    "interferogram": {
        "subtract_flat_earth_phase": True,
        "include_coherence": True,
    },
    "topo_phase_removal": {
        "dem_name": DEM_NAME,
    },
    "subset": {
        "region": SUBSET_REGION,
        "copy_metadata": True,
    },
    "terrain_correction": {
        "dem_name": DEM_NAME,
        "pixel_spacing_in_meter": PIXEL_SPACING_M,
        "save_selected_source_band": True,
        "save_local_incidence_angle": True,
    },
    "topsar_coregistration": {
        "dem_name": DEM_NAME,
        "keep_graph": True,
    },
}

if not SUBSET_REGION:
    raise ValueError("Set SUBSET_REGION before running subset / terrain correction")

## Step 1: Prepare Master and Slave Independently

This runs `split_orbit` on each input product separately. `deburst` is applied after pairwise coregistration because ESD needs burst-level inputs.

In [5]:
pair = PairProducts(master=MASTER_PRODUCT, slave=SLAVE_PRODUCT)
prepared_master = OUTPUT_DIR / "stepwise" / "master" / f"{MASTER_PRODUCT.stem}_ORB.dim"
prepared_slave = OUTPUT_DIR / "stepwise" / "slave" / f"{SLAVE_PRODUCT.stem}_ORB.dim"

if prepared_master.exists() and prepared_slave.exists():
    prepared_pair = PairProducts(master=prepared_master, slave=prepared_slave)
    print("Reusing prepared split/orbit products")
else:
    prepared_pair = prepare_pair(
        pair=pair,
        outdir=OUTPUT_DIR / "stepwise",
        preprocess_graphs=("split_orbit",),
        gpt_path=GPT_PATH.as_posix(),
        snap_userdir=SNAP_USERDIR,
        overrides=COMMON_OVERRIDES,
    )

print("Prepared master:", prepared_pair.master)
print("Prepared slave :", prepared_pair.slave)

Reusing prepared split/orbit products
Prepared master: /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/master/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB.dim
Prepared slave : /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/slave/S1C_IW_SLC__1SDV_20260208T122502_20260208T122529_006264_00C961_E47A_ORB.dim


## Step 2: Pairwise Coregistration

This is the multi-input workaround: `snapflow` writes a temporary XML graph that feeds both the prepared master and slave products into `Back-Geocoding`, then into `Enhanced-Spectral-Diversity`.

In [6]:
pair_gpt = GPT(
    product=prepared_pair.master,
    outdir=OUTPUT_DIR / "stepwise" / "pair",
    format="BEAM-DIMAP",
    gpt_path=GPT_PATH.as_posix(),
    snap_userdir=SNAP_USERDIR,
)

coreg_target = pair_gpt.outdir / "pair_coreg.dim"
if coreg_target.exists():
    coreg_path = coreg_target.as_posix()
    pair_gpt.prod_path = coreg_target
    print("Reusing pair coregistration output")
else:
    coreg_path = pair_gpt.topsar_coregistration(
        master_product=prepared_pair.master,
        slave_product=prepared_pair.slave,
        output_name="pair_coreg",
        **COMMON_OVERRIDES["topsar_coregistration"],
    )
    if coreg_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())
coreg_graph = pair_gpt.outdir / "pair_coreg_graph.xml"

print("Coregistered stack:", coreg_path)
print("Generated graph:", coreg_graph)
print(coreg_graph.read_text(encoding="utf-8")[:2000])

Reusing pair coregistration output
Coregistered stack: /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/pair_coreg.dim
Generated graph: /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/pair_coreg_graph.xml
<graph id="topsar-coregistration">
  <version>1.0</version>
  <node id="ReadMaster">
    <operator>Read</operator>
    <sources/>
    <parameters class="com.bc.ceres.binding.dom.XppDomElement">
      <file>${master}</file>
    </parameters>
  </node>
  <node id="ReadSlave">
    <operator>Read</operator>
    <sources/>
    <parameters class="com.bc.ceres.binding.dom.XppDomElement">
      <file>${slave}</file>
    </parameters>
  </node>
  <node id="BackGeocoding">
    <operator>Back-Geocoding</operator>
    <sources>
      <sourceProduct refid="ReadMaster"/>
      <sourceProduct.1 refid="ReadSlave"/>
    </sources>
    <parameters class="com.bc.ceres.binding.dom.XppDomElement">
      <demName>Copernicus 30m Glob

## Step 3: Continue the StaMPS-Prep Chain

In [7]:
deburst_target = pair_gpt.outdir / f"{pair_gpt.name}_DEB.dim"
if deburst_target.exists():
    deburst_path = deburst_target.as_posix()
    pair_gpt.prod_path = deburst_target
    print("Reusing deburst output")
else:
    deburst_path = pair_gpt.deburst()
    if deburst_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())

ifg_target = pair_gpt.outdir / f"{pair_gpt.name}_IFG.dim"
if ifg_target.exists():
    ifg_path = ifg_target.as_posix()
    pair_gpt.prod_path = ifg_target
    print("Reusing interferogram output")
else:
    ifg_path = pair_gpt.interferogram(**COMMON_OVERRIDES["interferogram"])
    if ifg_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())

topo_target = pair_gpt.outdir / f"{pair_gpt.name}_TOPO.dim"
if topo_target.exists():
    topo_path = topo_target.as_posix()
    pair_gpt.prod_path = topo_target
    print("Reusing topo-phase-removal output")
else:
    topo_path = pair_gpt.topo_phase_removal(**COMMON_OVERRIDES["topo_phase_removal"])
    if topo_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())

subset_target = pair_gpt.outdir / f"{pair_gpt.name}_SUB.dim"
if subset_target.exists():
    subset_path = subset_target.as_posix()
    pair_gpt.prod_path = subset_target
    print("Reusing subset output")
else:
    subset_path = pair_gpt.subset(**COMMON_OVERRIDES["subset"])
    if subset_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())

terrain_target = pair_gpt.outdir / f"{pair_gpt.name}_TC.dim"
if terrain_target.exists():
    terrain_path = terrain_target.as_posix()
    pair_gpt.prod_path = terrain_target
    print("Reusing terrain-correction output")
else:
    terrain_path = pair_gpt.terrain_correction(**COMMON_OVERRIDES["terrain_correction"])
    if terrain_path is None:
        raise RuntimeError(pair_gpt.last_error_summary())

print(json.dumps({
    "coregistration": coreg_path,
    "deburst": deburst_path,
    "interferogram": ifg_path,
    "topo_phase_removal": topo_path,
    "subset": subset_path,
    "terrain_correction": terrain_path,
}, indent=2))

Reusing deburst output
************************************************************************************************************************
OPERATOR: Interferogram
************************************************************************************************************************
Executing GPT command: /shared/home/rdelprete/PythonProjects/srp/snap13/bin/gpt -J-Dsnap.userdir=/shared/home/rdelprete/PythonProjects/srp/.snap -q 14 -x -e -SsourceProduct=/shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_DEB.dim Interferogram -PsubtractFlatEarthPhase=true -PsrpPolynomialDegree=5 -PsrpNumberPoints=501 -PorbitDegree=3 -PincludeCoherence=true -PcohWinAz=10 -PcohWinRg=10 -PsquarePixel=true -PsubtractTopographicPhase=false -PdemName="SRTM 3Sec" -PexternalDEMNoDataValue=0.0 -PexternalDEMApplyEGM=true -PtileExtensionPercent=100 -PoutputFlatEarthPhase=false -PoutputTopoPhase=false -Pou

GPT Output: Executing operator...
20%....30%....40%....50%....60%....70%....80%....90%.... done.
Writing...
.10%.20%....30%....40%....50%....60%....70%....80%....90%.... done.

GPT Warnings: INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.snap.core.gpf.common.WriteOp: Start writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_DEB_Ifg to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_IFG.dim
INFO: org.esa.snap.core.gpf.common.WriteOp: End writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_IFG to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_2026

GPT Output: Executing operator...
20%....30%....40%....50%....60%....70%....80%....90%.... done.
Writing...
.10%.20%....30%....40%....50%....60%....70%....80%....90%.... done.

GPT Warnings: INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.snap.core.gpf.common.WriteOp: Start writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_IFG_DInSAR to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_TOPO.dim
INFO: org.esa.snap.core.gpf.common.WriteOp: End writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_TOPO to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553

GPT Output: Executing operator...
20%....30%....40%....50%....60%....70%....80%....90%.... done.
Writing...
.10%.20%....30%32%32%.....44%......56%.....68%....78%....88%....98% done.

GPT Warnings: INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.snap.core.gpf.common.WriteOp: Start writing product Subset_S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_TOPO to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_SUB.dim
INFO: org.esa.snap.core.gpf.common.WriteOp: End writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_SUB to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T1

GPT Output: Executing operator...
20%....30%....40%....50%....60%.....72%.....84%.....96%.. done.
Writing...
..10%...21%21%.....33%....43%....53%....63%....73%....83%....93%... done.

GPT Warnings: INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.snap.core.gpf.common.WriteOp: Start writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_SUB_TC to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_TC.dim
INFO: org.esa.snap.core.gpf.common.WriteOp: End writing product S1A_IW_SLC__1SDV_20260226T122553_20260226T122620_063390_07F662_91CE_ORB_TC to /shared/home/rdelprete/PythonProjects/srp/data/output/insar_stamps_prep/stepwise/pair/S1A_IW_SLC__1SDV_20260226T122553_

## Optional: Run the Same Chain Through the Workflow Helper

In [8]:
RUN_WORKFLOW_HELPER = False

if RUN_WORKFLOW_HELPER:
    workflow_output = run_pair_workflow(
        pair=pair,
        outdir=OUTPUT_DIR / "workflow",
        workflow="stamps_prep",
        gpt_path=GPT_PATH.as_posix(),
        snap_userdir=SNAP_USERDIR,
        overrides=COMMON_OVERRIDES,
    )
    print("Workflow output:", workflow_output)
else:
    print("Skipping workflow helper rerun. Set RUN_WORKFLOW_HELPER=True to execute it.")

Skipping workflow helper rerun. Set RUN_WORKFLOW_HELPER=True to execute it.
